# DOT Interest-Rate Scenario Calculator

Notebook version of `dot_scenario_calculator.py` (converted -- the
original script is removed once this notebook is confirmed to reproduce
its numbers). Layers a documented, user-supplied interest-rate assumption
on top of the validated baseline forecast from `dot_forecast_model.ipynb`.
This notebook does **not** change `dot_forecast_model.ipynb` or re-fit
anything -- it only reuses its `predict()` / `train_models()` functions
and does simple arithmetic on top of their output.

### The core distinction this notebook exists to keep visible

> **BASELINE forecast** comes from `dot_forecast_model`'s `predict(year)`.
> Model-derived: fit on FY2017-2024 actuals, backtested against
> FY2025-2027. That is a real, validated number.
>
> **INTEREST-RATE EFFECT** is a number **you** supply
> (`effect_per_25bps`). It is a **documented planning assumption, not a
> learned coefficient** -- it is NOT fit to any data, NOT learned by the
> model, and NOT validated by a backtest. It's laid on top of a validated
> number in its own function, applied as plain arithmetic, so it's always
> possible to see exactly what the model produced vs. what a human
> assumption added on top.

## Reusing `dot_forecast_model.ipynb`'s functions

Three ways to reuse another notebook's code were considered:

1. **A plain Python `import`** -- not possible anymore now that
   `dot_forecast_model` is a notebook, not a `.py` module.
2. **IPython's `%run` magic** -- works in a real Jupyter kernel, but this
   project's execution environment sometimes runs notebooks through a
   plain-Python executor (no IPython layer) for reliability, where `%run`
   is not valid syntax at all. Using it here would make this notebook
   fail to execute in that path.
3. **Read the sibling notebook's code cells directly (via `nbformat`) and
   `exec` them into this notebook's namespace** -- pure Python, works
   identically under a real Jupyter kernel or a plain-Python executor.
   **This is what's used below.**

`__name__` is deliberately set to `"dot_forecast_model"` (not
`"__main__"`) before executing those cells -- this mirrors exactly what a
real `import dot_forecast_model` would have done, so that notebook's own
`if __name__ == "__main__":`-guarded demo cell is correctly skipped here
(we want its function *definitions*, not its demo output).

In [1]:
import nbformat


def run_notebook_as_module(path: str, module_name: str, namespace: dict) -> None:
    '''Execute a sibling notebook's code cells into `namespace`, with
    __name__ set to `module_name` (not "__main__") -- mirrors what a real
    `import` does, so any `if __name__ == "__main__":`-guarded cell in the
    source notebook (its own demo/driver code) is correctly skipped, the
    same way it would be on a real Python import.
    '''
    namespace["__name__"] = module_name
    nb = nbformat.read(path, as_version=4)
    for cell in nb.cells:
        if cell.cell_type == "code":
            exec(compile(cell.source, path, "exec"), namespace)


run_notebook_as_module("dot_forecast_model.ipynb", "dot_forecast_model", globals())
print("Loaded load_data, train_models, predict, backtest from dot_forecast_model.ipynb")

Loaded load_data, train_models, predict, backtest from dot_forecast_model.ipynb


## The Assumption -- Read This Before Trusting Any Number Below

`DEFAULT_EFFECT_PER_25BPS` is a **planning assumption, not a regression
coefficient.** Nothing in `dot_forecast_model` estimates how DOT's budget
responds to interest rates -- there is no interest-rate column anywhere
in the training data (`raw datasets/DOT_Budget_2017_2027.csv`,
`DOT_Spending_Merged.csv`). This number is picked by a human and stated
here in the open, not derived.

**Direction assumed (documented, not derived):** a rate CUT is modeled as
a modest INCREASE to budget need, on the illustrative theory that cheaper
borrowing costs make more capital projects economically viable and DOT
takes on more of them. That's one plausible story, not the only one --
flip the sign of `effect_per_25bps` if you want to model the opposite
assumption (e.g. rate cuts easing financing costs and reducing need).

**Why the effect is likely small for DOT specifically:** DOT is an
operating agency -- its budget here is operating Adopted/Modified/Actual
dollars, not municipal debt issuance. Interest rates bite hardest on
**debt service**, which for NYC sits on a separate citywide debt-service
budget line, not inside an individual operating agency's budget like
DOT's. This scenario tool models a secondary, assumption-driven effect,
not a primary, data-driven one -- see the disclaimer at the end.

In [2]:
DEFAULT_EFFECT_PER_25BPS = 0.5  # percent, per 25 bps of rate cut -- ASSUMPTION, not learned

## `scenario()`

Baseline forecast (from the validated model) + a documented interest-rate
scenario adjustment (from a stated assumption, not data). Returns the
baseline, the assumption applied, the resulting dollar adjustment, and
the scenario total -- kept as **separate fields on purpose**, never
pre-summed away, so it's always possible to see the model's number and
the assumption's contribution separately.

In [3]:
def scenario(year: int, basis_points_cut: float,
             effect_per_25bps: float = DEFAULT_EFFECT_PER_25BPS,
             series: str = "modified") -> dict:
    '''
    year : fiscal year to forecast (passed straight to predict())
    basis_points_cut : size of the assumed rate cut, in basis points.
        Positive = a cut. Use a negative number to model a hike instead.
    effect_per_25bps : ASSUMPTION, not learned -- % budget change per 25 bps
        of cut. Defaults to DEFAULT_EFFECT_PER_25BPS; override to test other
        assumptions.
    series : which of the model's three series to apply this to
        ("adopted", "modified", or "actual"). Defaults to "modified", DOT's
        primary planning-budget figure.
    '''
    baseline_row = predict(year).loc[series]
    baseline = baseline_row["point_forecast"]

    # The assumption, applied as plain arithmetic -- no fitting, no data.
    pct_adjustment = (basis_points_cut / 25) * effect_per_25bps  # percent
    adjustment_amount = baseline * (pct_adjustment / 100)
    scenario_value = baseline + adjustment_amount

    return {
        "year": year,
        "series": series,
        "basis_points_cut": basis_points_cut,
        "effect_per_25bps": effect_per_25bps,
        "baseline": baseline,
        "baseline_lower_95": baseline_row["lower_95"],
        "baseline_upper_95": baseline_row["upper_95"],
        "pct_adjustment": pct_adjustment,
        "adjustment_amount": adjustment_amount,
        "scenario_value": scenario_value,
    }

## `print_scenario_table()`

Prints a clean baseline / adjustment / scenario table for one year across
a list of basis-point-cut scenarios (e.g. `[0, 10, 25]`).

In [4]:
def print_scenario_table(year: int, bps_scenarios, series: str = "modified",
                          effect_per_25bps: float = DEFAULT_EFFECT_PER_25BPS) -> None:
    train_models()  # ensure a model is trained/saved before predicting

    print(f"\nFY{year} {series.capitalize()} budget -- interest-rate scenarios")
    print(f"(assumption: {effect_per_25bps:+.2f}% per 25bps of rate cut)\n")

    header = f"{'Scenario':<18}{'Baseline':>18}{'Adjustment':>16}{'Scenario':>18}"
    print(header)
    print("-" * len(header))

    for bps in bps_scenarios:
        r = scenario(year, bps, effect_per_25bps=effect_per_25bps, series=series)
        label = "No cut (baseline)" if bps == 0 else f"{bps:.0f} bps cut"
        print(
            f"{label:<18}{r['baseline']:>18,.0f}{r['adjustment_amount']:>16,.0f}"
            f"{r['scenario_value']:>18,.0f}"
        )

    baseline_row = predict(year).loc[series]
    print(
        f"\nFor reference, the model's own 95% prediction interval on the "
        f"FY{year} baseline (before any scenario adjustment):"
        f"\n  [{baseline_row['lower_95']:,.0f}, {baseline_row['upper_95']:,.0f}]"
    )
    print(
        "Scenario adjustments above are typically a small fraction of that "
        "interval's width -- worth comparing the two directly rather than "
        "reading the scenario number as more precise than the baseline it's "
        "built on."
    )

## Disclaimer

In [5]:
DISCLAIMER = '''
================================================================================
DISCLAIMER -- read this before using any number above for planning
================================================================================
BASELINE figures (the "Baseline" column) are MODEL-DERIVED: a linear trend
fit on FY2017-2024 actual DOT budget data, backtested against FY2025-2027.
That is a real, validated number (see dot_forecast_model.ipynb's own
backtest() output for the exact error/coverage figures).

INTEREST-RATE ADJUSTMENTS (the "Adjustment" and resulting "Scenario"
columns) are NOT model-derived. They come from a single stated planning
assumption (effect_per_25bps) with no supporting regression, no fitted
coefficient, and no backtest of its own -- there is no interest-rate data
anywhere in this project's inputs. Treat the adjustment as "what if we
assumed X," not as a forecast.

DOT is an operating agency. Its Adopted/Modified/Actual budget here is
day-to-day operating spending, not municipal debt issuance. Interest
rates bite hardest on DEBT SERVICE, which for NYC is a separate,
citywide budget line -- not inside an individual operating agency's
budget the way modeled here. Any interest-rate sensitivity applied to
DOT's own operating budget in this notebook is a secondary, illustrative
effect, not DOT's primary or best-documented cost driver.
================================================================================
'''

## Demo -- FY2028 Scenario Table

Plain, unconditional cell (no `__name__` guard) -- by this point in the
notebook, `__name__` was already set to `"dot_forecast_model"` by the
loader cell above, so a guard here would incorrectly skip this too. A
notebook's own final cell doesn't need one anyway; guards are only needed
on the *reused* notebook's demo code, not this one's.

In [6]:
print_scenario_table(2028, bps_scenarios=[0, 10, 25])
print(DISCLAIMER)

[load_data] Found cached table at ../data/dot_yearly.csv -- loading directly.
[load_data] Loaded 11 rows, FY2017-2027.

[train_models] Training window: FY2017-2024 (8 observations). Held out: FY2025-2027.

    series      slope ($/yr)           intercept      r2
   adopted     74,174,894.88  -148,711,528,370.46  0.9172
  modified     75,728,331.07  -151,814,784,660.32  0.9392
    actual     59,622,076.02  -119,412,282,258.98  0.8331

[train_models] Saved models to ../models/dot_forecast_models.pkl.

FY2028 Modified budget -- interest-rate scenarios
(assumption: +0.50% per 25bps of rate cut)

Scenario                    Baseline      Adjustment          Scenario
----------------------------------------------------------------------
No cut (baseline)      1,762,270,753               0     1,762,270,753
10 bps cut             1,762,270,753       3,524,542     1,765,795,294
25 bps cut             1,762,270,753       8,811,354     1,771,082,106

For reference, the model's own 95% prediction